# Experiment 08 — Direct GBM (no recursion)

Experiment 06 showed that *removing recursion* (linear direct per-series) loses to our recursive GBM — but that test also threw away the strong learner. Experiment 07 showed that recursively-recomputed dynamic features **hurt most on the longest, most test-like window**, hinting that recursion's error accumulation is itself a cost. This notebook isolates that one variable: keep the **strong pooled GBM**, but forecast **directly** — predict all 16 days at once from features known for the whole horizon, with **no feedback loop**.

**What "safe for direct forecasting" means.** A feature is usable only if, for every one of the 16 horizon days, it depends solely on data available at the forecast cutoff (the day before the window). So:

- **Kept:** all calendar / Fourier / holidays / geo / payday (known); `onpromotion` and oil with their moving averages (known-future covariates); `transactions_lag_16…23` (safe by construction); sales lags **≥16 days** (`lag_21, lag_28, lag_42, lag_56, lag_364, lag_365`).
- **Dropped:** every recursion-dependent feature — `lag_1…14`, all `rolling_mean_7/14/28/364` (they use `shift(1)`, i.e. the within-horizon days), and the dynamic features from experiment 07.
- **Added (safe recent level):** `safe_mean_16_22`, `safe_mean_16_43`, `safe_std_16_43`, `safe_dow` — means/volatility of sales **16–43 days back** and the same-weekday level (`lag_21`,`lag_28`). These give the model the "recent level" anchor that the austere linear model of experiment 06 lacked, while staying leak-free for the entire horizon.

Because nothing depends on within-horizon sales, the window/test days are predicted by a **single direct pass** (no iterative loop). Trade-off: we lose the very strong short lags (`lag_1`, `lag_7`), but gain zero error accumulation — and, crucially, a model that is **maximally different** from our recursive GBM, which is the real hope for the **blend**. Gate unchanged: beat `geo_blend25` (public 0.38899) on all 3 windows.

> 🇷🇺 Прямой GBM без рекурсии: тот же сильный пулинговый learner, но прогноз всех 16 дней сразу — только из признаков, известных на весь горизонт. Оставлены календарь/Fourier/праздники/гео/payday, `onpromotion` и нефть (известное будущее), `transactions_lag_16…23`, лаги продаж **≥16** (`lag_21…365`). Выброшены все рекурсивные (`lag_1…14`, `rolling_mean_*` через shift(1), динамические фичи 07). Добавлены **безопасные агрегаты недавнего уровня** (`safe_mean_16_22/16_43`, `safe_std_16_43`, `safe_dow`) — среднее/волатильность за 16–43 дня назад, чего не хватало аскетичной линейной модели 06. Прогноз — один прямой проход, без цикла. Минус: теряем сильные короткие лаги; плюс: ноль накопления ошибки и модель, максимально отличная от нашей рекурсивной — главная надежда на **бленд**. Гейт прежний: побить `geo_blend25` (0.38899) на всех 3 окнах.

In [1]:
import time
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, lightgbm as lgb, joblib

In [2]:
_t0 = time.time()

MODELS_DIR = "../models/exp08"; ART_DIR = "../artifacts/exp08"; EXP05_ART = "../artifacts/exp05"
os.makedirs(MODELS_DIR, exist_ok=True); os.makedirs(ART_DIR, exist_ok=True)

PARAMS = dict(n_estimators=4155, learning_rate=0.01, num_leaves=255, min_child_samples=30,
              colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
              reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1, verbose=-1)

# Safe feature set: everything usable for direct forecasting (no within-horizon dependence)
# Безопасный набор: всё, что применимо для прямого прогноза (без зависимости от дней внутри горизонта)
SAFE_PARQUET = [
    "day_of_week", "month", "year", "is_weekend", "day_of_year", "week_of_year", "date_index",
    "sin_day", "cos_day", "sin_week", "cos_week",
    "onpromotion", "onpromotion_ma7", "onpromotion_ma28",
    "dcoilwtico", "dcoilwtico_ma7", "dcoilwtico_ma28",
    *[f"transactions_lag_{l}" for l in range(16, 24)],
    "lag_21", "lag_28", "lag_42", "lag_56", "lag_364", "lag_365",     # sales lags >= 16 days
    "is_holiday_national", "day_before_holiday", "is_black_friday", "is_cyber_monday", "is_terremoto",
    "is_navidad", "is_dia_madre", "is_futbol", "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day",
    "is_holiday_local", "is_holiday_regional",
    "day_of_month", "is_payday", "days_since_payday", "days_to_payday",
    "store_nbr", "store_type", "cluster", "family", "city", "state",
]
SAFE_COMPUTED = ["safe_mean_16_22", "safe_mean_16_43", "safe_std_16_43", "safe_dow"]
FEATURES = SAFE_PARQUET + SAFE_COMPUTED
CAT = ["store_nbr", "store_type", "cluster", "family", "day_of_week", "month", "city", "state"]

WINDOWS = [("W1", pd.Timestamp("2017-06-15"), pd.Timestamp("2017-06-30")),
           ("W2", pd.Timestamp("2017-07-15"), pd.Timestamp("2017-07-30")),
           ("W3", pd.Timestamp("2017-07-31"), pd.Timestamp("2017-08-15"))]
WNAMES = [w for w, _, _ in WINDOWS]
TEST_START, TEST_END = pd.Timestamp("2017-08-16"), pd.Timestamp("2017-08-31")
print(f"{len(FEATURES)} safe features ({len(SAFE_PARQUET)} parquet + {len(SAFE_COMPUTED)} computed)")

59 safe features (55 parquet + 4 computed)


In [3]:
# Load parquet, reshape sales, compute the safe recent-level aggregates (shift >= 16, leak-free)
# Загружаем parquet, матрица продаж, безопасные агрегаты недавнего уровня (shift >= 16, без утечки)
DF = (pd.read_parquet("../artifacts/features.parquet")
      .sort_values(["date", "store_nbr", "family"]).reset_index(drop=True))
DATES = np.sort(DF["date"].unique()); N_DATES = len(DATES); N_SERIES = DF.groupby("date").size().iloc[0]
assert len(DF) == N_DATES * N_SERIES
DT = pd.to_datetime(DATES); DATE_IDX = {d: i for i, d in enumerate(DATES)}
ref = DF.iloc[:N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
SALES = DF["sales"].to_numpy(float).reshape(N_DATES, N_SERIES)

m1622 = np.full((N_DATES, N_SERIES), np.nan); m1643 = np.full((N_DATES, N_SERIES), np.nan)
s1643 = np.full((N_DATES, N_SERIES), np.nan); dow = np.full((N_DATES, N_SERIES), np.nan)
for i in range(N_DATES):
    if i >= 22:
        m1622[i] = np.nanmean(SALES[i - 22:i - 15], axis=0)            # mean of sales 16..22 days back
    if i >= 43:
        seg = SALES[i - 43:i - 15]; c = (~np.isnan(seg)).sum(0)        # 16..43 days back
        m1643[i] = np.nanmean(seg, axis=0)
        ok = c >= 2
        if ok.any():
            s1643[i, ok] = np.nanstd(seg[:, ok], axis=0, ddof=1)
    if i >= 28:
        dow[i] = np.nanmean(SALES[[i - 21, i - 28]], axis=0)           # same-weekday level (>=16 back)

FEAT = DF[SAFE_PARQUET].copy()
FEAT["safe_mean_16_22"] = m1622.reshape(-1)
FEAT["safe_mean_16_43"] = m1643.reshape(-1)
FEAT["safe_std_16_43"]  = s1643.reshape(-1)
FEAT["safe_dow"]        = dow.reshape(-1)
for col in CAT:
    FEAT[col] = FEAT[col].astype("category")
DATE_ARR = DF["date"].to_numpy()
print(f"{N_DATES} dates x {N_SERIES} series; FEAT {FEAT.shape}")

1704 dates x 1782 series; FEAT (3036528, 59)


In [4]:
# Direct forecasting: train once per cutoff, predict the 16 window/test days in a single pass (no loop)
# Прямой прогноз: обучение на отсечке, предсказание 16 дней одним проходом (без цикла)
def rmsle_mat(P, Y):
    return float(np.sqrt(np.mean((np.log1p(np.clip(P, 0, None)) - np.log1p(Y)) ** 2)))


def zero_rule_mask(first_idx):
    return np.nansum(SALES[first_idx - 21:first_idx], axis=0) == 0


def rows_for_window(win_idxs):
    idx = np.concatenate([np.arange(i * N_SERIES, (i + 1) * N_SERIES) for i in win_idxs])
    return idx


def train_direct(cutoff, tag):
    path = os.path.join(MODELS_DIR, f"direct_{tag}.joblib")
    if os.path.exists(path):
        return joblib.load(path)
    mask = (DATE_ARR >= np.datetime64("2016-01-01")) & (DATE_ARR < np.datetime64(cutoff)) & DF["sales"].notna().to_numpy()
    X = FEAT[mask]; y = np.log1p(DF.loc[mask, "sales"])
    t = time.time(); m = lgb.LGBMRegressor(**PARAMS); m.fit(X, y); joblib.dump(m, path)
    print(f"  direct_{tag}: {mask.sum():,} rows, {time.time()-t:.0f}s"); return m


def predict_direct(model, win_idxs, zmask):
    idx = rows_for_window(win_idxs)
    P = np.expm1(model.predict(FEAT.iloc[idx])).clip(0).reshape(len(win_idxs), N_SERIES)
    P[:, zmask] = 0.0
    return P


def logblend(A, B, w=0.5):
    return np.expm1(w * np.log1p(np.clip(A, 0, None)) + (1 - w) * np.log1p(np.clip(B, 0, None)))


# Sanity: safe features must not depend on within-horizon sales — check safe_mean_16_43 on a window day
# uses only data >=16 days back (i.e. before the window starts)
# Проверка: безопасные фичи не зависят от продаж внутри горизонта; используются только данные >=16 дней назад
i = DATE_IDX[pd.Timestamp("2017-08-10")]  # a W3 day
assert np.isnan(SALES[i]).sum() == 0  # actuals exist (it's train)
manual = np.nanmean(SALES[i - 43:i - 15], axis=0)
stored = FEAT["safe_mean_16_43"].to_numpy().reshape(N_DATES, N_SERIES)[i]
assert np.allclose(manual, stored, equal_nan=True), "safe agg mismatch"
print("OK: safe aggregates use only data >=16 days back")

OK: safe aggregates use only data >=16 days back


In [5]:
# Main runs: direct GBM, and its log-space blend with geo_blend25
# Основные прогоны: прямой GBM и его log-бленд с geo_blend25
RESULTS, rows, WIN_Y = {}, [], {}
for wname, ws, we in WINDOWS:
    print(f"\n=== {wname}: {ws.date()} .. {we.date()} ===")
    win_idxs = [i for i in range(N_DATES) if ws <= DT[i] <= we]
    assert len(win_idxs) == 16
    Y = SALES[win_idxs]; zmask = zero_rule_mask(win_idxs[0]); WIN_Y[wname] = Y
    geo_blend25 = np.load(os.path.join(EXP05_ART, f"pred_{wname}__geo_blend25.npy"))

    m = train_direct(ws, wname)
    P_direct = predict_direct(m, win_idxs, zmask)
    P_blend = logblend(P_direct, geo_blend25, 0.5)
    RESULTS[(wname, "direct_gbm")] = P_direct
    RESULTS[(wname, "geo_blend25")] = geo_blend25
    RESULTS[(wname, "direct_blend")] = P_blend
    for sch in ["geo_blend25", "direct_gbm", "direct_blend"]:
        sc = rmsle_mat(RESULTS[(wname, sch)], Y)
        rows.append({"window": wname, "scheme": sch, "rmsle": sc})
        print(f"  {sch:14s} RMSLE={sc:.5f}")

res = pd.DataFrame(rows); res.to_csv(os.path.join(ART_DIR, "results.csv"), index=False)
print(f"\nElapsed: {(time.time()-_t0)/60:.1f} min")


=== W1: 2017-06-15 .. 2017-06-30 ===
  geo_blend25    RMSLE=0.38173
  direct_gbm     RMSLE=0.38353
  direct_blend   RMSLE=0.37965

=== W2: 2017-07-15 .. 2017-07-30 ===
  geo_blend25    RMSLE=0.38652
  direct_gbm     RMSLE=0.40225
  direct_blend   RMSLE=0.38933

=== W3: 2017-07-31 .. 2017-08-15 ===
  geo_blend25    RMSLE=0.40187
  direct_gbm     RMSLE=0.40321
  direct_blend   RMSLE=0.39749

Elapsed: 0.3 min


In [6]:
# Summary and gate
piv = res.pivot(index="scheme", columns="window", values="rmsle")
piv["mean"] = piv[WNAMES].mean(axis=1); piv = piv.sort_values("mean")
display(piv.round(5))

def beats(c, base="geo_blend25"):
    return all(piv.loc[c, w] < piv.loc[base, w] for w in WNAMES)

print("\nvs geo_blend25 (gate = win all 3 windows):")
for c in ["direct_gbm", "direct_blend"]:
    print(f"  {c:14s} " + "  ".join(f"{w}:{piv.loc[c, w]-piv.loc['geo_blend25', w]:+.5f}" for w in WNAMES),
          "  gate=" + ("PASS" if beats(c) else "fail"))
GATED = [c for c in ["direct_gbm", "direct_blend"] if beats(c)]
FINAL = min(GATED, key=lambda c: piv.loc[c, "mean"]) if GATED else None
print("\nGate passed by:", GATED if GATED else "NOBODY", "| FINAL:", FINAL)

window,W1,W2,W3,mean
scheme,,,,
direct_blend,0.37965,0.38933,0.39749,0.38882
geo_blend25,0.38173,0.38652,0.40187,0.39004
direct_gbm,0.38353,0.40225,0.40321,0.39633



vs geo_blend25 (gate = win all 3 windows):
  direct_gbm     W1:+0.00180  W2:+0.01573  W3:+0.00134   gate=fail
  direct_blend   W1:-0.00207  W2:+0.00281  W3:-0.00439   gate=fail

Gate passed by: NOBODY | FINAL: None


In [7]:
# Blend-weight sensitivity
# The 50/50 blend fails the gate on W2, but the direct model is best used as a SMALL diverse dose
# (it is weaker and noisier alone, so it should not dominate the blend). Sweep the weight.
# Чувствительность к весу бленда
# Бленд 50/50 проваливает W2, но direct лучше использовать малой разнообразящей дозой
# (сам он слабее и шумнее — не должен доминировать). Свип по весу.
gb = {wn: rmsle_mat(RESULTS[(wn, "geo_blend25")], WIN_Y[wn]) for wn in WNAMES}
print(f"{'w_direct':>8} | " + "  ".join(f"{w:>9}" for w in WNAMES) + f" | {'mean':>8} | gate")
sweep = {}
for w in [0.15, 0.20, 0.25, 0.30, 0.35, 0.50]:
    sc = {wn: rmsle_mat(logblend(RESULTS[(wn, "direct_gbm")], RESULTS[(wn, "geo_blend25")], w), WIN_Y[wn]) for wn in WNAMES}
    sweep[w] = sc
    d = {wn: sc[wn] - gb[wn] for wn in WNAMES}
    g = "PASS" if all(d[wn] < 0 for wn in WNAMES) else "fail"
    print(f"{w:>8.2f} | " + "  ".join(f"{d[wn]:+8.5f}" for wn in WNAMES) + f" | {np.mean(list(sc.values())):.5f} | {g}")

# Among weights that beat geo_blend25 on ALL 3 windows, pick the best mean / Среди весов, бьющих на всех 3 окнах, берём лучшее среднее
PASSW = [w for w in sorted(sweep) if all(sweep[w][wn] < gb[wn] for wn in WNAMES)]
BLEND_W = min(PASSW, key=lambda w: np.mean(list(sweep[w].values()))) if PASSW else None
print(f"\nGate-passing weights: {PASSW} -> chosen BLEND_W = {BLEND_W}")

w_direct |        W1         W2         W3 |     mean | gate
    0.15 | -0.00125  -0.00025  -0.00237 | 0.38875 | PASS
    0.20 | -0.00154  -0.00012  -0.00296 | 0.38850 | PASS
    0.25 | -0.00178  +0.00011  -0.00345 | 0.38833 | fail
    0.30 | -0.00196  +0.00044  -0.00384 | 0.38825 | fail
    0.35 | -0.00208  +0.00088  -0.00413 | 0.38826 | fail
    0.50 | -0.00207  +0.00281  -0.00439 | 0.38882 | fail

Gate-passing weights: [0.15, 0.2] -> chosen BLEND_W = 0.2


In [8]:
# Final: refit on full train, predict test, build the direct-blend submission
# Финал: переобучение на всём train, прогноз теста, сборка сабмишна direct-бленда
test_idxs = [i for i in range(N_DATES) if TEST_START <= DT[i] <= TEST_END]
m_full = train_direct(TEST_START, "FULL")
P_direct_test = predict_direct(m_full, test_idxs, zero_rule_mask(test_idxs[0]))
np.save(os.path.join(ART_DIR, "pred_TEST__direct_gbm.npy"), P_direct_test)
test = pd.read_csv("../data/test.csv", parse_dates=["date"])
long = pd.concat([pd.DataFrame({"date": DATES[i], "store_nbr": ref["store_nbr"].values,
                                "family": ref["family"].values, "sales": P_direct_test[j]})
                  for j, i in enumerate(test_idxs)], ignore_index=True)
direct_sub = test.merge(long, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]].sort_values("id").reset_index(drop=True)
geo = pd.read_csv("../submission_05_geo_blend25.csv").sort_values("id").reset_index(drop=True)
assert (direct_sub["id"] == geo["id"]).all()

if BLEND_W is None:
    print("No blend weight beats geo_blend25 on all 3 windows — no submission.")
else:
    out = geo[["id"]].copy()
    out["sales"] = logblend(direct_sub["sales"].to_numpy(), geo["sales"].to_numpy(), BLEND_W)
    out.to_csv("../submission_08_direct_blend20.csv", index=False)
    print(f"Saved submission_08_direct_blend20.csv (w_direct={BLEND_W}, total {out['sales'].sum():,.0f})")
print(f"\nTotal runtime: {(time.time()-_t0)/60:.1f} min")

Saved submission_08_direct_blend20.csv (w_direct=0.2, total 12,091,643)

Total runtime: 0.4 min


## Conclusions

`direct_gbm` alone is weaker than the recursive GBM (mean 0.39633 vs 0.39004) — as expected, it gives up the strong short lags (`lag_1`, `lag_7`). But it is very different from the recursive model, so a small dose helps in the blend. A 50/50 blend fails the gate; the weight sweep shows the right amount is small:

| w_direct | W1 | W2 | W3 | gate |
|---:|---:|---:|---:|:--|
| **0.20** | **−0.00154** | **−0.00012** | **−0.00296** | **PASS** |
| 0.25 | −0.00178 | +0.00011 | −0.00345 | fail |
| 0.50 | −0.00207 | +0.00281 | −0.00439 | fail |

(Δ vs `geo_blend25`; negative = better.) At **w=0.20** the blend beats `geo_blend25` on all three windows on **validation**.

### ⚠️ The validation gain did NOT transfer

| Submission | Validation (mean) | **Public** | vs best |
|---|---:|---:|---:|
| deployed best mix (`mix_geo_blend25`) | — | **0.38803** | — |
| `submission_08_direct_blend20` | 0.38850 | **0.39417** | +0.006 |
| `submission_mix_direct_blend20` | — | **0.38956** | +0.0015 |

A 0.20 dose improved validation but worsened the public score by **0.006**. The direct GBM does well on W3 (Aug 1–15) but poorly on the true test (Aug 16–31): same month, different half. The second half of August likely carries effects the direct model captures worse (end-of-month payday, the back-to-school ramp) — invisible to a validation that stops on Aug 15.

**The main lesson of the project, shown three times** (the experiment-04 hybrid and both experiment-08 submissions): here, validation gains smaller than ~0.002–0.003 RMSLE do not reliably transfer to the public leaderboard, even with an honest, leak-free, multi-window gate. The only changes that transferred were larger and structural — the geo features (notebook 05) and the file-level mix. **The practical ceiling of this honest single-pipeline approach is ≈0.388**; the ~0.3735 "wall" is a heavily-forked tuned ensemble, not a target for single-model work. The deployed best stays the mix at **0.38803**; `submission_08_direct_blend20` is not deployed.

## Выводы

`direct_gbm` сам по себе слабее рекурсивного GBM (среднее 0.39633 против 0.39004) — ожидаемо, он теряет сильные короткие лаги (`lag_1`, `lag_7`). Но он сильно отличается от рекурсивной модели, поэтому малая доза помогает в бленде. Бленд 50/50 проваливает гейт; свип веса показывает, что правильная доза мала:

| w_direct | W1 | W2 | W3 | гейт |
|---:|---:|---:|---:|:--|
| **0.20** | **−0.00154** | **−0.00012** | **−0.00296** | **PASS** |
| 0.25 | −0.00178 | +0.00011 | −0.00345 | fail |
| 0.50 | −0.00207 | +0.00281 | −0.00439 | fail |

(Δ относительно `geo_blend25`; отрицательное = лучше.) При **w=0.20** бленд бьёт `geo_blend25` на всех трёх окнах на **валидации**.

### ⚠️ Выигрыш с валидации НЕ перенёсся

| Сабмишн | Валидация (среднее) | **Паблик** | против лучшего |
|---|---:|---:|---:|
| лучший боевой микс (`mix_geo_blend25`) | — | **0.38803** | — |
| `submission_08_direct_blend20` | 0.38850 | **0.39417** | +0.006 |
| `submission_mix_direct_blend20` | — | **0.38956** | +0.0015 |

Доза 0.20 улучшила валидацию, но ухудшила публичный скор на **0.006**. Direct GBM хорош на W3 (1–15 авг), но плох на реальном тесте (16–31 авг): тот же месяц, другая половина. Вторая половина августа, вероятно, несёт эффекты, которые direct ловит хуже (зарплата конца месяца, школьный сезон) — невидимые валидации, которая заканчивается 15 августа.

**Главный урок проекта, показанный трижды** (гибрид из эксперимента 04 и оба сабмита эксперимента 08): здесь выигрыши на валидации меньше ~0.002–0.003 RMSLE не переносятся на публичный лидерборд надёжно, даже при честном многооконном гейте без утечек. Перенеслось только крупное и структурное — geo-фичи (ноутбук 05) и файловый микс. **Практический потолок честного однопайплайнового подхода ≈0.388**; «стена» ~0.3735 — это растиражированный тюненый ансамбль, а не цель для одномодельной работы. Лучший боевой результат остаётся миксом **0.38803**; `submission_08_direct_blend20` не деплоится.